# SPD Responding Units Data Cleaning & Preparation
**Project:** Investigating Crisis Call Volumes — Eugene vs. Springfield  
**Author:** Ilhan Haniff  
**Description:** This notebook imports, cleans, and standardizes the Springfield Police Department (SPD) Responding Units Excel file (2015–2025), constructs temporal and unit classification features, and exports a single combined cleaned dataset for analysis.

**Input:** `data/2015-2025_SPD_Responding_Units.xlsx` (one sheet per year)  **Output:** `data/spd_clean_2015_2025.csv`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

## 2. Load Designator Notes
The first sheet contains unit classification reference lists. I'll load these to build our classification logic.

In [6]:
filepath = os.path.join('data', '2015-2025_SPD_Responding_Units.xlsx')

# Load designator notes sheet
designators = pd.read_excel(filepath, sheet_name='Designator Notes', engine='openpyxl')
print('Designator Notes columns:', list(designators.columns))
designators.head(16)

Designator Notes columns: ['Patrol Designators', 'Detective Designators', 'CSOs and Animal Control', 'Other/Command Staff', 'Not SPD', 'CAHOOTS']


,Patrol Designators,Detective Designators,CSOs and Animal Control,Other/Command Staff,Not SPD,CAHOOTS
0,3D1,4S40,5C1,CMD02,30000000000000001826021443431825408,3J81
1,2S12,4S41,5C2,CMD03,2W71,NaN
2,1S12,4S42,5C4,CMD04,2Y19,NaN
3,1S13,4S43,5C5,CMD05,3V10,NaN
4,1S22,4S44,NaN,CMD6,3X33,NaN
5,1S18,4S46,NaN,2S50,CNTR,NaN
6,1S21,4S47,NaN,2S60,UO327,NaN
7,1S11,4S48,NaN,NaN,Z23,NaN
8,2S18,4S49,NaN,NaN,Z24,NaN
9,2S21,NaN,NaN,NaN,Z25,NaN


In [5]:
# Extract unit lists from each category column and clean them
def extract_unit_list(series):
    """Extract non-null, stripped unit identifiers from a designator column."""
    return set(series.dropna().astype(str).str.strip().str.upper())

patrol_units   = extract_unit_list(designators['Patrol Designators'])
detective_units = extract_unit_list(designators['Detective Designators'])
cso_ac_units   = extract_unit_list(designators['CSOs and Animal Control'])
command_units  = extract_unit_list(designators['Other/Command Staff'])
not_spd_units  = extract_unit_list(designators['Not SPD'])

print(f'Patrol units:     {len(patrol_units)}')
print(f'Detective units:  {len(detective_units)}')
print(f'CSO/AC units:     {len(cso_ac_units)}')
print(f'Command units:    {len(command_units)}')
print(f'Not SPD units:    {len(not_spd_units)}')

Patrol units:     56
Detective units:  9
CSO/AC units:     4
Command units:    7
Not SPD units:    16


## 3. Load and Combine All Yearly Sheets

In [7]:
years = range(2015, 2026)
raw_frames = []

for year in years:
    df = pd.read_excel(filepath, sheet_name=str(year), engine='openpyxl')
    df['source_year'] = year  # track which sheet each row came from
    raw_frames.append(df)
    print(f'{year}: {df.shape[0]:,} rows loaded')

spd_raw = pd.concat(raw_frames, ignore_index=True)
print(f'\nTotal combined rows: {spd_raw.shape[0]:,}')s
print(f'Columns: {list(spd_raw.columns)}')

2015: 54,804 rows loaded
2016: 56,210 rows loaded
2017: 57,033 rows loaded
2018: 54,653 rows loaded
2019: 54,774 rows loaded
2020: 50,026 rows loaded
2021: 50,009 rows loaded
2022: 49,921 rows loaded
2023: 48,737 rows loaded
2024: 47,780 rows loaded
2025: 43,708 rows loaded

Total combined rows: 567,655
Columns: ['inci id', 'Unnamed: 1', 'call time', 'prime unit', 'units dispatched in order', 'source_year']


## 4. Initial Inspection

In [9]:
spd_raw.head(-5)

,inci id,Unnamed: 1,call time,prime unit,units dispatched in order,source_year
0,15000004,NaN,01/01/2015 00:01,1S12,1S12,2015
1,15000013,NaN,01/01/2015 00:04,,NaN,2015
2,15000022,NaN,01/01/2015 00:11,3S12,"4S73 ,3S12 ,3S18",2015
3,15000026,NaN,01/01/2015 00:14,1S12,1S12,2015
4,15000029,NaN,01/01/2015 00:17,1S18,1S18,2015
...,...,...,...,...,...,...
567645,25326519,NaN,12/30/2025 22:24,3J81,3J81,2025
567646,25326528,NaN,12/30/2025 22:34,1S21,"1S21 ,1S22",2025
567647,25326529,NaN,12/30/2025 22:39,1S13,"1S11 ,1S13",2025
567648,25326532,NaN,12/30/2025 22:49,,NaN,2025


In [10]:
spd_raw.dtypes

inci id                        int64
Unnamed: 1                   float64
call time                     object
prime unit                    object
units dispatched in order     object
source_year                    int64
dtype: object

In [11]:
spd_raw.isnull().sum()

inci id                           0
Unnamed: 1                   567655
call time                         0
prime unit                        0
units dispatched in order     76834
source_year                       0
dtype: int64

## 5. Standardize Column Names

In [12]:
# Standardize column names: lowercase, replace spaces with underscores
spd_raw.columns = spd_raw.columns.str.strip().str.lower().str.replace(' ', '_')
print('Standardized columns:', list(spd_raw.columns))

Standardized columns: ['inci_id', 'unnamed:_1', 'call_time', 'prime_unit', 'units_dispatched_in_order', 'source_year']


In [15]:
# Keep only relevant columns
cols_to_keep = ['inci_id', 'call_time', 'prime_unit', 'source_year']
spd = spd_raw[cols_to_keep].copy()
print(f'Kept {len(cols_to_keep)} columns. Shape: {spd.shape}')
spd.head(-5)

Kept 4 columns. Shape: (567655, 4)


,inci_id,call_time,prime_unit,source_year
0,15000004,01/01/2015 00:01,1S12,2015
1,15000013,01/01/2015 00:04,,2015
2,15000022,01/01/2015 00:11,3S12,2015
3,15000026,01/01/2015 00:14,1S12,2015
4,15000029,01/01/2015 00:17,1S18,2015
...,...,...,...,...
567645,25326519,12/30/2025 22:24,3J81,2025
567646,25326528,12/30/2025 22:34,1S21,2025
567647,25326529,12/30/2025 22:39,1S13,2025
567648,25326532,12/30/2025 22:49,,2025


## 6. Clean Text Variables
Strip whitespace and standardize capitalization in string columns.

In [23]:
spd['prime_unit'] = spd['prime_unit'].astype(str).str.strip().str.upper()

# Re-encode empty strings as NaN (some blank cells may have loaded as whitespace)
spd['prime_unit'] = spd['prime_unit'].replace({'': np.nan, 'NAN': np.nan})

print('Sample prime_unit values:')
print(spd['prime_unit'].dropna().unique()[:1000])

Sample prime_unit values:
['1S12' '3S12' '1S18' '1S21' '1S20' '3D1' '1S11' '3D2' '4S73' '1S22'
 '3S13' '2S11' '2S21' '2S22' '2S12' '5C4' '3S11' '2S20' '3S23' '3S21'
 '3S22' '3S10' 'SP687' 'SP690' '1S13' '1S23' 'SP689' '4S72' 'SP679' '5C1'
 '2M4' 'SP628' 'SP649' 'SP632' '5C2' '1S19' 'SP692' '2S10' 'SP688' 'SP018'
 '1S17' '2M2' '5C6' 'SP655' 'SP674' 'SP328' '2S18' 'SP659' 'SP289' 'SP600'
 '3S18' '3S20' 'SP357' '4S49' '4S45' '4S43' '2S13' '2M1' 'SP640' 'SP297'
 '4S48' 'K98' '4S42' '3B55' 'SP621' 'SP680' 'SP381' '3J81' '4S40' 'SP602'
 'SP354' '1S10' '4S47' '2S23' '2S19' 'SP672' 'AWO' 'K94' 'SP359' 'SP339'
 '4S41' 'SP084' 'SP355' 'SP350' '5X51' '3D3' '1M13' 'SP076' '2Y45' '2E11'
 'K95' 'CAHOT' 'K97' 'CMD1' 'SP377' 'SP384' 'SP693' 'CMD01' '2C81' 'SP060'
 'SP344' '2M3' '1E52' 'SP376' '3S37' '3S31' 'SP338' 'SP263' 'SP310'
 'CMD02' 'SP284' 'CMD04' 'CMD05' 'CMD03' '3J78' 'K92' '1M14' '3S35'
 'SP362' '3S38' 'SP410' 'SP656' 'SP304' 'SP422' '3S34' '3S32' '3S39'
 '3S36' 'SP365' '3S19' 'SP325' 'CMD06

## 7. Convert call_time to Datetime and Extract Temporal Features

In [27]:
spd['calltime'] = pd.to_datetime(spd['calltime'], errors='coerce')

# Extract temporal features
spd['call_year']  = spd['calltime'].dt.year
spd['call_month'] = spd['calltime'].dt.month
spd['call_day']   = spd['calltime'].dt.day
spd['call_hour']  = spd['calltime'].dt.hour
spd['call_dow']   = spd['calltime'].dt.day_name()

print('Temporal features created.')
spd.head(-5)

Temporal features created.


,inci_id,prime_unit,source_year,calltime,call_year,call_month,call_day,call_hour,call_dow
0,15000004,1S12,2015,2015-01-01 00:01:00,2015,1,1,0,Thursday
1,15000013,NaN,2015,2015-01-01 00:04:00,2015,1,1,0,Thursday
2,15000022,3S12,2015,2015-01-01 00:11:00,2015,1,1,0,Thursday
3,15000026,1S12,2015,2015-01-01 00:14:00,2015,1,1,0,Thursday
4,15000029,1S18,2015,2015-01-01 00:17:00,2015,1,1,0,Thursday
...,...,...,...,...,...,...,...,...,...
567645,25326519,3J81,2025,2025-12-30 22:24:00,2025,12,30,22,Tuesday
567646,25326528,1S21,2025,2025-12-30 22:34:00,2025,12,30,22,Tuesday
567647,25326529,1S13,2025,2025-12-30 22:39:00,2025,12,30,22,Tuesday
567648,25326532,NaN,2025,2025-12-30 22:49:00,2025,12,30,22,Tuesday


## 8. Classify Unit Type
Using the Designator Notes sheet, classify each responding unit into a category.

Classification rules (in order):
- Missing `prime_unit` → `UNKNOWN`
- Unit in CSO/Animal Control list → `CSO_AC`
- Unit in Detective list → `DETECTIVE`
- Unit in Not SPD list OR contains `'E'` (per designator notes) → `NOT_SPD`
- Unit in Command Staff list → `COMMAND`
- Unit in Patrol list OR remaining SPD → `PATROL`
- Anything else → `OTHER`

In [33]:
def classify_spd_unit(unit):
    """
    Classifies an SPD prime_unit identifier into a unit type category.
    Returns one of: 'PATROL', 'DETECTIVE', 'CSO_AC', 'COMMAND', 'NOT_SPD', 'OTHER', 'UNKNOWN'
    """
    # Missing unit
    if pd.isna(unit) or unit == '':
        return 'UNKNOWN'

    unit = str(unit).strip().upper()

    if unit in cso_ac_units:
        return 'CSO_AC'

    if unit in detective_units:
        return 'DETECTIVE'

    # Per designator notes: anything with an 'E' (e.g. 2E14) is Not SPD
    if unit in not_spd_units or 'E' in unit:
        return 'NOT_SPD'

    if unit in command_units:
        return 'COMMAND'

    if unit in patrol_units:
        return 'PATROL'

    return 'OTHER'

spd['unit_type'] = spd['prime_unit'].apply(classify_spd_unit)
spd = spd.drop(columns="source_year")


print('unit_type value counts:')
print(spd['unit_type'].value_counts())
spd.head(-5)

unit_type value counts:
unit_type
PATROL       360946
OTHER         89370
UNKNOWN       79587
CSO_AC        34200
DETECTIVE      3281
COMMAND         209
NOT_SPD          62
Name: count, dtype: int64


,inci_id,prime_unit,calltime,call_year,call_month,call_day,call_hour,call_dow,unit_type
0,15000004,1S12,2015-01-01 00:01:00,2015,1,1,0,Thursday,PATROL
1,15000013,NaN,2015-01-01 00:04:00,2015,1,1,0,Thursday,UNKNOWN
2,15000022,3S12,2015-01-01 00:11:00,2015,1,1,0,Thursday,PATROL
3,15000026,1S12,2015-01-01 00:14:00,2015,1,1,0,Thursday,PATROL
4,15000029,1S18,2015-01-01 00:17:00,2015,1,1,0,Thursday,PATROL
...,...,...,...,...,...,...,...,...,...
567645,25326519,3J81,2025-12-30 22:24:00,2025,12,30,22,Tuesday,OTHER
567646,25326528,1S21,2025-12-30 22:34:00,2025,12,30,22,Tuesday,PATROL
567647,25326529,1S13,2025-12-30 22:39:00,2025,12,30,22,Tuesday,PATROL
567648,25326532,NaN,2025-12-30 22:49:00,2025,12,30,22,Tuesday,UNKNOWN


## 9. Handle Missing Values and Duplicates

In [34]:
# Check duplicates
dupes = spd.duplicated(subset=['inci_id']).sum()
print(f'Duplicate inci_id rows: {dupes}')

# Missing values
print('\nMissing values per column:')
print(spd.isnull().sum())

Duplicate inci_id rows: 0

Missing values per column:
inci_id           0
prime_unit    79587
calltime          0
call_year         0
call_month        0
call_day          0
call_hour         0
call_dow          0
unit_type         0
dtype: int64


## 10. Final Dataset Summary

In [35]:
print('=== Final Cleaned SPD Dataset Summary ===')
print(f'Shape: {spd.shape}')
print(f'Date range: {spd["calltime"].min()} to {spd["calltime"].max()}')
print(f'\nColumns: {list(spd.columns)}')
print(f'\nunit_type distribution:')
print(spd['unit_type'].value_counts())
print(f'\nCalls per year:')
print(spd.groupby('call_year').size())

=== Final Cleaned SPD Dataset Summary ===
Shape: (567655, 9)
Date range: 2015-01-01 00:01:00 to 2025-12-30 23:49:00

Columns: ['inci_id', 'prime_unit', 'calltime', 'call_year', 'call_month', 'call_day', 'call_hour', 'call_dow', 'unit_type']

unit_type distribution:
unit_type
PATROL       360946
OTHER         89370
UNKNOWN       79587
CSO_AC        34200
DETECTIVE      3281
COMMAND         209
NOT_SPD          62
Name: count, dtype: int64

Calls per year:
call_year
2015    54804
2016    56210
2017    57033
2018    54653
2019    54774
2020    50026
2021    50009
2022    49921
2023    48737
2024    47780
2025    43708
dtype: int64


In [36]:
spd.head(5)

,inci_id,prime_unit,calltime,call_year,call_month,call_day,call_hour,call_dow,unit_type
0,15000004,1S12,2015-01-01 00:01:00,2015,1,1,0,Thursday,PATROL
1,15000013,NaN,2015-01-01 00:04:00,2015,1,1,0,Thursday,UNKNOWN
2,15000022,3S12,2015-01-01 00:11:00,2015,1,1,0,Thursday,PATROL
3,15000026,1S12,2015-01-01 00:14:00,2015,1,1,0,Thursday,PATROL
4,15000029,1S18,2015-01-01 00:17:00,2015,1,1,0,Thursday,PATROL


## 11. Export Cleaned Dataset

In [37]:
output_path = os.path.join('data', 'spd_clean_2015_2025.csv')
spd.to_csv(output_path, index=False)
print(f'Cleaned dataset saved to: {output_path}')
print(f'Final shape: {spd.shape}')

Cleaned dataset saved to: data/spd_clean_2015_2025.csv
Final shape: (567655, 9)
